In [13]:
import os
import os.path as op
from collections import OrderedDict
import pandas as pd
import numpy as np
import shutil
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
# Define the main directory and target directory paths
deriv_dir = "./derivatives/none-reduced-no-motion"
reg_dir = os.path.join(deriv_dir, "regression")

## Create the dataframes for the Regression Analysis (physical health, rsFC)

#### Sig Dimensions:
##### Exluding the none network : Dim 1, 3


In [28]:
#make sure to consider if you want modified/orginial phy health variables
phyhealth_df = pd.read_csv(os.path.join(reg_dir, "phyhealth-reg-mod.csv"))

In [29]:
phyhealth_df

,src_subject_id,BMI,mctq_sdweek_calc,sleep_chrono,physical_activity1_y,cbcl_scr_syn_internal_t,cbcl_scr_syn_external_t,delta_weight,blood_pressure_mean,resp_composite
0,NDAR_INV030W95VP,30.209996,9.3677,neither,2.0,62.0,40.0,stable,79.166667,0.0
1,NDAR_INV0DC9BJZK,22.292121,6.7774,morning,7.0,63.0,34.0,stable,68.333333,0.0
2,NDAR_INV0DKWEM1A,30.716425,7.8534,morning,2.0,34.0,40.0,gain,91.555556,0.0
3,NDAR_INV0MPBK7TU,18.015334,8.8526,evening,3.0,52.0,51.0,stable,73.666667,0.0
4,NDAR_INV0RHLKA9M,23.193343,9.8785,neither,7.0,66.0,64.0,loss,91.833333,0.0
...,...,...,...,...,...,...,...,...,...,...
166,NDAR_INVZM2Y9JCA,13.754639,8.1008,neither,2.0,40.0,48.0,stable,73.444444,0.0
167,NDAR_INVZP49GXF4,20.982317,7.6798,evening,7.0,47.0,48.0,stable,73.555556,0.0
168,NDAR_INVZR9NMJBR,16.834671,9.0393,neither,5.0,44.0,34.0,stable,76.333333,0.0
169,NDAR_INVZT1J0KUC,22.657497,7.7963,neither,3.0,50.0,44.0,stable,93.166667,0.0


In [30]:
rsfc_df = pd.read_csv(os.path.join(deriv_dir, "rsfc-sub.csv"))
sociocult_df = pd.read_csv(os.path.join(deriv_dir, "sociocult_Nan.csv"))
covariate_df = pd.read_csv(os.path.join(deriv_dir, "covariate.csv"))


In [31]:
# we need to add back the subject id column so we remove the correct rows in the next step

# x is rsfc, y is sociocult
latent_df = pd.read_csv(op.join(deriv_dir, "rniXrsfc_lx-base.csv"))

latent_df["src_subject_id"] = rsfc_df["src_subject_id"].values

print(latent_df)

     Unnamed: 0            V1            V2            V3            V4  \
0             1  1.820539e-01  2.504047e-03 -1.512469e-01  1.120020e-01   
1             2 -6.608028e-02  1.235386e-01  2.596251e-01 -1.384371e-03   
2             3 -1.285239e-01  2.794798e-02 -4.424122e-02 -2.470388e-03   
3             4  7.141278e-02 -3.731526e-02  4.575467e-03  1.634960e-01   
4             5 -7.457920e-03 -1.676335e-01 -1.510913e-01 -3.347016e-02   
..          ...           ...           ...           ...           ...   
231         232  2.297232e-01  6.972639e-02  2.702908e-01  1.257131e-01   
232         233  1.581743e-15 -8.767131e-18 -1.269438e-15  1.120771e-15   
233         234 -1.791623e-02  1.767558e-01  2.776197e-02 -8.635549e-02   
234         235  1.780701e-01  2.068012e-01  3.621239e-01 -4.566530e-02   
235         236  8.387936e-02  8.855848e-02 -4.260417e-02 -4.022700e-02   

               V5            V6            V7            V8            V9  \
0    7.737539e-02 -9.6

In [36]:
latent_df = latent_df[latent_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
sociocult_df = sociocult_df[sociocult_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
covariate_df = covariate_df[covariate_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]
rsfc_df = rsfc_df[rsfc_df["src_subject_id"].isin(phyhealth_df["src_subject_id"])]

In [37]:
print(rsfc_df)

       src_subject_id  rsfmri_c_ngd_ad_ngd_ad  rsfmri_c_ngd_ad_ngd_cgc  \
1    NDAR_INV030W95VP                0.424266                 0.320737   
2    NDAR_INV0DC9BJZK                0.261305                 0.121087   
3    NDAR_INV0DKWEM1A                0.269920                 0.148276   
6    NDAR_INV0MPBK7TU                0.300166                 0.193031   
7    NDAR_INV0RHLKA9M                0.325872                 0.201443   
..                ...                     ...                      ...   
231  NDAR_INVZM2Y9JCA                0.227564                 0.064915   
232  NDAR_INVZP49GXF4                0.277297                 0.159869   
233  NDAR_INVZR9NMJBR                0.224798                 0.130237   
234  NDAR_INVZT1J0KUC                0.187419                 0.096684   
235  NDAR_INVZYRTFYRP                0.231318                 0.133982   

     rsfmri_c_ngd_ad_ngd_ca  rsfmri_c_ngd_ad_ngd_dt  rsfmri_c_ngd_ad_ngd_dla  \
1                  0.125510    

In [38]:
phyhealth_reg_save_path = os.path.join(reg_dir, "phyhealth-reg-mod.csv")
rsfc_reg_save_path = os.path.join(reg_dir, "rsfc-reg.csv")
latent_reg_save_path = os.path.join(reg_dir, "latent-reg.csv")
covariate_reg_save_path = os.path.join(reg_dir, "covariate-reg.csv")

phyhealth_df.to_csv(phyhealth_reg_save_path, index=False)
rsfc_df.to_csv(rsfc_reg_save_path, index=False)
latent_df.to_csv(latent_reg_save_path, index=False)
covariate_df.to_csv(covariate_reg_save_path, index=False)

print("All CSVs saved successfully!")

All CSVs saved successfully!


In [39]:
# create df for the corr coeff reg analysis
# rsfc measures that came back as signficant
# will make a df for each of these that has all of the covariates and phy health measures

# Define dimension 1 measures
dim1_rsfc_reg_measures = [
    #"rsfmri_c_ngd_cgc_ngd_dt",
    #"rsfmri_c_ngd_dt_ngd_dla",
    "rsfmri_c_ngd_dt_ngd_dt",
    #"rsfmri_c_ngd_dt_ngd_vs",
    "rsfmri_c_ngd_vs_ngd_vs",
]
dim1_rsfc_short_labels = [
    #"cgc-dt", 
    #"dt-dla", 
    "DN-DN", 
    #"DN-VN", 
    "VN-VN"]

# Define dimension 3 measures
dim3_rsfc_reg_measures = [
    "rsfmri_c_ngd_dt_ngd_smm",
    #"rsfmri_c_ngd_vta_ngd_vs",
]
dim3_rsfc_short_labels = [
    "DN-SMN", 
    #"vta-vs"
]

# Process dim1 measures
dim1_dir = op.join(reg_dir, "dim1")
os.makedirs(dim1_dir, exist_ok=True)

for measure, short_label in zip(dim1_rsfc_reg_measures, dim1_rsfc_short_labels):
    # 1) pull out subj‑ID + measure, then rename to "rsfc"
    tmp = rsfc_df[["src_subject_id", measure]].copy()
    tmp.rename(columns={measure: "rsfc"}, inplace=True)

    # 2) merge in covariates + phys‑health
    tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
    tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")

    # 3) write out
    out_path = os.path.join(dim1_dir, f"phyhealth_{short_label}_data.csv")
    tmp.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")

# Process dim3 measures
dim3_dir = op.join(reg_dir, "dim3")
os.makedirs(dim3_dir, exist_ok=True)

for measure, short_label in zip(dim3_rsfc_reg_measures, dim3_rsfc_short_labels):
    # 1) pull out subj‑ID + measure, then rename to "rsfc"
    tmp = rsfc_df[["src_subject_id", measure]].copy()
    tmp.rename(columns={measure: "rsfc"}, inplace=True)

    # 2) merge in covariates + phys‑health
    tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
    tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")

    # 3) write out
    out_path = os.path.join(dim3_dir, f"phyhealth_{short_label}_data.csv")
    tmp.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")

Wrote ./derivatives/none-reduced-no-motion/regression/dim1/phyhealth_DN-DN_data.csv
Wrote ./derivatives/none-reduced-no-motion/regression/dim1/phyhealth_VN-VN_data.csv
Wrote ./derivatives/none-reduced-no-motion/regression/dim3/phyhealth_DN-SMN_data.csv


In [40]:
# create df for the latent score reg analysis
# latent dimensions that came back as signficant
# will make a df for each of these that has all of the covariates and phy health measures

# Process dim1 latent scores
dim1_dir = op.join(reg_dir, "dim1")
os.makedirs(dim1_dir, exist_ok=True)

tmp = latent_df[["src_subject_id", "V1"]].copy()
tmp.rename(columns={"V1": "score"}, inplace=True)
tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")
out_path = os.path.join(dim1_dir, "phyhealth_dim1_latent_data.csv")
tmp.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

# Process dim3 latent scores
dim3_dir = op.join(reg_dir, "dim3")
os.makedirs(dim3_dir, exist_ok=True)

tmp = latent_df[["src_subject_id", "V3"]].copy()
tmp.rename(columns={"V3": "score"}, inplace=True)
tmp = tmp.merge(covariate_df, on="src_subject_id", how="inner")
tmp = tmp.merge(phyhealth_df, on="src_subject_id", how="inner")
out_path = os.path.join(dim3_dir, "phyhealth_dim3_latent_data.csv")
tmp.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote ./derivatives/none-reduced-no-motion/regression/dim1/phyhealth_dim1_latent_data.csv
Wrote ./derivatives/none-reduced-no-motion/regression/dim3/phyhealth_dim3_latent_data.csv
